# 大模型处理中文诗句的完整流程 Demo

展示 Tokenization → Embedding → Self-Attention 的全过程

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## 第一步：模拟 Tokenization（BPE 风格）

In [2]:
class SimpleTokenizer:
    """模拟 BPE tokenizer 的行为"""

    def __init__(self):
        # 简化的词表：单字 + 常见词组
        # 真实模型的词表通常有 50k-100k 个 token
        self.vocab = {
            # 特殊 token
            "[PAD]": 0, "[UNK]": 1, "[CLS]": 2, "[SEP]": 3,
            # 单字
            "山": 4, "重": 5, "水": 6, "复": 7, "疑": 8, "无": 9, "路": 10,
            "柳": 11, "暗": 12, "花": 13, "明": 14, "又": 15, "一": 16, "村": 17,
            "，": 18, "。": 19,
            # 常见词组（BPE 会合并高频组合）
            "山重": 20, "水复": 21, "柳暗": 22, "花明": 23, "又一": 24, "一村": 25,
        }
        self.id_to_token = {v: k for k, v in self.vocab.items()}

    def tokenize(self, text: str) -> list[str]:
        """
        模拟 BPE 分词：
        1. 优先匹配长词组（合并的 token）
        2. 无法匹配时拆分为单字
        """
        tokens = []
        i = 0
        while i < len(text):
            # 尝试匹配 2 字词组
            if i + 1 < len(text):
                bigram = text[i:i+2]
                if bigram in self.vocab:
                    tokens.append(bigram)
                    i += 2
                    continue
            # 回退到单字
            tokens.append(text[i])
            i += 1
        return tokens

    def encode(self, text: str) -> list[int]:
        """文本 → Token IDs"""
        tokens = self.tokenize(text)
        return [self.vocab.get(t, self.vocab["[UNK]"]) for t in tokens]

    def decode(self, ids: list[int]) -> str:
        """Token IDs → 文本"""
        return "".join(self.id_to_token.get(i, "[UNK]") for i in ids)

## 第二步：Embedding 层

In [3]:
class EmbeddingLayer(nn.Module):
    """
    Token Embedding + Position Embedding
    """

    def __init__(self, vocab_size: int, d_model: int, max_seq_len: int = 512):
        super().__init__()
        # Token Embedding：每个 token ID 映射到一个 d_model 维向量
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        # Position Embedding：位置信息
        self.position_embedding = nn.Embedding(max_seq_len, d_model)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        """
        输入: [batch_size, seq_len] 的 token IDs
        输出: [batch_size, seq_len, d_model] 的嵌入向量
        """
        seq_len = token_ids.size(1)
        positions = torch.arange(seq_len, device=token_ids.device)

        token_emb = self.token_embedding(token_ids)      # [batch, seq, d_model]
        pos_emb = self.position_embedding(positions)     # [seq, d_model]

        return token_emb + pos_emb  # 广播相加

## 第三步：Self-Attention（核心！）

Self-Attention：让每个 token 关注序列中的所有其他 token
这是"柳暗花明"能够被整体理解的关键！

In [4]:
class SelfAttention(nn.Module):
    """
    Self-Attention：让每个 token 关注序列中的所有其他 token
    这是"柳暗花明"能够被整体理解的关键！
    """

    def __init__(self, d_model: int, n_heads: int = 4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        assert d_model % n_heads == 0, "d_model 必须能被 n_heads 整除"

        # Q, K, V 投影矩阵
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        输入/输出: [batch_size, seq_len, d_model]
        返回: (输出, 注意力权重)
        """
        batch_size, seq_len, _ = x.shape

        # 计算 Q, K, V
        Q = self.W_q(x)  # [batch, seq, d_model]
        K = self.W_k(x)
        V = self.W_v(x)

        # 重塑为多头: [batch, n_heads, seq, head_dim]
        Q = Q.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        # 注意力分数: Q @ K^T / sqrt(d_k)
        # [batch, n_heads, seq, seq]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        # Softmax 归一化（得到注意力权重矩阵）
        attention_weights = F.softmax(scores, dim=-1)

        # 加权求和: Attention @ V
        # [batch, n_heads, seq, head_dim]
        context = torch.matmul(attention_weights, V)

        # 合并多头: [batch, seq, d_model]
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        # 输出投影
        output = self.W_o(context)

        return output, attention_weights

## 第四步：完整的 Transformer Block

In [5]:
class TransformerBlock(nn.Module):
    """一个完整的 Transformer 层"""

    def __init__(self, d_model: int, n_heads: int, d_ff: int = 256):
        super().__init__()
        self.attention = SelfAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # Feed-Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # Self-Attention + 残差连接 + LayerNorm
        attn_out, attn_weights = self.attention(x)
        x = self.norm1(x + attn_out)

        # FFN + 残差连接 + LayerNorm
        x = self.norm2(x + self.ffn(x))

        return x, attn_weights

## 完整 Demo：端到端流程

In [6]:
# 配置
d_model = 64    # 嵌入维度（真实模型通常是 768-4096）
n_heads = 4     # 注意力头数
vocab_size = 26 # 词表大小

# 1. 初始化各层
tokenizer = SimpleTokenizer()
embedding_layer = EmbeddingLayer(vocab_size, d_model)
transformer = TransformerBlock(d_model, n_heads)

# 2. 输入诗句
text = "山重水复疑无路，柳暗花明又一村"
print(f"📝 原始输入: {text}")

# 3. Tokenization
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)
print(f"\n🔪 Tokenization 结果:")
print(f"   Tokens: {tokens}")
print(f"   Token IDs: {token_ids}")

# 4. 转换为 Tensor
input_tensor = torch.tensor([token_ids])  # [1, seq_len]
print(f"\n🔢 输入 Tensor shape: {input_tensor.shape}")

# 5. Embedding
embeddings = embedding_layer(input_tensor)
print(f"\n📊 Embedding shape: {embeddings.shape}")
print(f"   (batch_size=1, seq_len={len(token_ids)}, d_model={d_model})")

# 6. Self-Attention
output, attention_weights = transformer(embeddings)
print(f"\n🧠 Transformer 输出 shape: {output.shape}")
print(f"   注意力权重矩阵 shape: {attention_weights.shape}")

📝 原始输入: 山重水复疑无路，柳暗花明又一村

🔪 Tokenization 结果:
   Tokens: ['山重', '水复', '疑', '无', '路', '，', '柳暗', '花明', '又一', '村']
   Token IDs: [20, 21, 8, 9, 10, 18, 22, 23, 24, 17]

🔢 输入 Tensor shape: torch.Size([1, 10])

📊 Embedding shape: torch.Size([1, 10, 64])
   (batch_size=1, seq_len=10, d_model=64)

🧠 Transformer 输出 shape: torch.Size([1, 10, 64])
   注意力权重矩阵 shape: torch.Size([1, 4, 10, 10])


## 注意力权重可视化

In [7]:
# 可视化注意力权重（简化版）
print(f"🔍 注意力权重可视化（第1个头，前5个token）:")
attn = attention_weights[0, 0, :5, :5].detach().numpy()
tokens_5 = tokens[:5]

# 打印表头
print("     ", end="")
for t in tokens_5:
    print(f"{t:>6}", end="")
print()

# 打印每行
for i, t in enumerate(tokens_5):
    print(f"{t:>4}", end="")
    for j in range(5):
        print(f"{attn[i,j]:>6.2f}", end="")
    print()

🔍 注意力权重可视化（第1个头，前5个token）:
         山重    水复     疑     无     路
  山重  0.03  0.07  0.13  0.09  0.38
  水复  0.14  0.12  0.09  0.10  0.02
   疑  0.08  0.10  0.13  0.11  0.08
   无  0.03  0.09  0.12  0.11  0.28
   路  0.10  0.03  0.06  0.07  0.10


## 关键理解

1. **Tokenization 不是传统"分词"，而是 BPE 等子词算法**
   - 高频词组会被合并成单个 token（如"柳暗"、"花明"）
   - 低频词会被拆成单字

2. **Embedding 为每个 token 生成向量表示**
   - 初始是静态的，由查表得到
   - 加上位置编码后包含顺序信息

3. **Self-Attention 是语义理解的核心！**
   - 每个字的向量会与所有其他字交互
   - 注意力权重决定了"关注程度"
   - "柳"会强烈关注"暗"、"花"、"明"，形成整体语义
   - 这就是为什么模型能理解"柳暗花明"的隐喻含义

4. **多层 Transformer 堆叠后，形成深层语义表示**
   - 可以捕捉复杂的语义关系和上下文